<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study/11_Fashion_Product_Images_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 가벼운 다이어트 버전 다운로드 (500MB 대)
!kaggle datasets download -d paramaggarwal/fashion-product-images-small

# 2. 압축 해제
!unzip -q fashion-product-images-small.zip

Dataset URL: https://www.kaggle.com/datasets/paramaggarwal/fashion-product-images-small
License(s): MIT
100% 565M/565M [00:06<00:00, 88.8MB/s]



In [ ]:
!ls -l

total 584124
-rw-r--r-- 1 root root 592614936 Oct 22  2019 fashion-product-images-small.zip
drwxr-xr-x 2 root root   1175552 Jun  6 11:18 images
drwxr-xr-x 3 root root      4096 Jun  6 11:18 myntradataset
drwxr-xr-x 1 root root      4096 Jun  4 13:39 sample_data
-rw-r--r-- 1 root root   4332000 Oct 22  2019 styles.csv


In [ ]:
import pandas as pd

df = pd.read_csv('styles.csv', on_bad_lines='skip')
# print(df.head())
df['id'] = df['id'].astype(str)

print(len(df))

df[['id', 'gender', 'masterCategory', 'subCategory', 'baseColour', 'season']].head()

44424


,id,gender,masterCategory,subCategory,baseColour,season
0,15970,Men,Apparel,Topwear,Navy Blue,Fall
1,39386,Men,Apparel,Bottomwear,Blue,Summer
2,59263,Women,Accessories,Watches,Silver,Winter
3,21379,Men,Apparel,Bottomwear,Black,Fall
4,53759,Men,Apparel,Topwear,Grey,Summer


In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.metrics import f1_score
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# 1. 전역 시드 제어 (재현성 확보)
SEED = 42
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

# 2. 전역 인프라 설정 팩토리
CONFIG = {
    'img_size': 224,
    'batch_size': 128,
    'epochs': 3,
    'lr': 3e-4,
    'n_splits': 5,
    'model_name': 'resnet18',
    'img_dir': 'images',
    'csv_path': 'styles.csv'
}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("device:", device)

device: cuda


In [ ]:
# CSV 데이터 로드 (에러 라인 패스 옵션 적용)
df = pd.read_csv(CONFIG['csv_path'], on_bad_lines='skip')
df['id'] = df['id'].astype(str)

# 결측치가 있으면 학습이 불가능하므로 사전에 Drop
df = df.dropna(subset=['gender', 'masterCategory', 'subCategory', 'baseColour', 'season'])

# 대상 텍스트 컬럼 정의
label_columns = ['gender', 'masterCategory', 'subCategory', 'baseColour', 'season']

# 고유 태그 사전(Vocabulary) 빌드
all_tags = []
for col in label_columns:
    all_tags.extend(df[col].unique().tolist())
unique_tags = sorted(list(set(all_tags)))

tag_to_idx = {tag: idx for idx, tag in enumerate(unique_tags)}
idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}
NUM_CLASSES = len(unique_tags)

print(NUM_CLASSES)

# Multi Hot Encoding 매핑 함수
def encode_multi_hot(row, tag_map, num_classes):
    vector = np.zeros(num_classes, dtype=np.float32)
    for col in label_columns:
        tag = row[col]
        if tag in tag_map:
            vector[tag_map[tag]] = 1.0
    return vector

# 데이터프레임 -> Multi Hot Matrix 변환 수행
targets = np.array([encode_multi_hot(row, tag_to_idx, NUM_CLASSES) for _, row in df.iterrows()])
print("전처리 완료 shape:", targets.shape)

105
전처리 완료 shape: (44388, 105)


In [ ]:
class FashionMultiLabelDataset(Dataset):
    def __init__(self, df, targets, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.targets = targets
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")

        # 파일이 비어있거나 소실되었을 때의 방어 코드 (NPE 방지)
        if not os.path.exists(img_path):
            image = np.zeros((CONFIG['img_size'], CONFIG['img_size'], 3), dtype=np.uint8)
        else:
            from PIL import Image
            image = np.array(Image.open(img_path).convert('RGB'))

        target = self.targets[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, torch.tensor(target, dtype=torch.float32)

train_transforms = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.05, rotate_limit=15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(CONFIG['img_size'], CONFIG['img_size']),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

In [ ]:
class MultiLabelModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.model = timm.create_model(model_name, pretrained=pretrained)

        # backbone 모델의 수많은 분류기 종류(fc, classifier, head)에 모두 대응하는 동적 인젝션 코드
        if hasattr(self.model, 'fc'):
            in_features = self.model.fc.in_features
            self.model.fc = nn.Linear(in_features, num_classes)
        elif hasattr(self.model, 'classifier'):
            in_features = self.model.classifier.in_features
            self.model.classifier = nn.Linear(in_features, num_classes)
        elif hasattr(self.model, 'head'):
            in_features = self.model.head.fc.in_features
            self.model.head.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # BCEWithLogitsLoss 내부에서 자체 Sigmoid 연산이 이루어지므로 raw logit 상태로 리턴합니다.
        return self.model(x)

print(f"{CONFIG['model_name']} 기반 다중 라벨 커스텀 모델 클래스 선언 완료")

resnet18 기반 다중 라벨 커스텀 모델 클래스 선언 완료


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, scheduler, device):
    model.train()
    train_loss = 0.0
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
    scheduler.step()
    return train_loss / len(dataloader.dataset)

def validate(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    all_targets = []
    all_preds = []

    with torch.no_grad():
        for images, targets in dataloader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            val_loss += loss.item() * images.size(0)

            # 시그모이드를 거쳐 각 태그별 독립 확률(0.0 ~ 1.0) 도출
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(targets.cpu().numpy())

    val_loss /= len(dataloader.dataset)
    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)

    # 다중 라벨 핵심 연산: 0.5 임계값 기준으로 0과 1 인코딩
    all_preds_bin = (all_preds > 0.5).astype(int)

    # 다중 라벨 특화 지표 (Macro F1, Sample F1) 계산
    macro_f1 = f1_score(all_targets, all_preds_bin, average='macro', zero_division=0)
    samples_f1 = f1_score(all_targets, all_preds_bin, average='samples', zero_division=0)

    return val_loss, macro_f1, samples_f1

In [ ]:
kf = KFold(n_splits=CONFIG['n_splits'], shuffle=True, random_state=SEED)
oof_macro_f1 = []
oof_samples_f1 = []

for fold, (train_idx, val_idx) in enumerate(kf.split(df)):
    print(f"\n" + "="*40 + f"\n FOLD {fold + 1} / {CONFIG['n_splits']} 가동\n" + "="*40)

    # 데이터 분할 추출
    train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]
    train_tgt, val_tgt = targets[train_idx], targets[val_idx]

    # 데이터셋 & 데이터로더 조립
    train_dataset = FashionMultiLabelDataset(train_df, train_tgt, CONFIG['img_dir'], transform=train_transforms)
    val_dataset = FashionMultiLabelDataset(val_df, val_tgt, CONFIG['img_dir'], transform=val_transforms)

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True, num_workers=4, drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, num_workers=4)

    # 인스턴스 초기화 및 다중 라벨 손실함수 (BCEWithLogitsLoss) 주입
    model = MultiLabelModel(model_name=CONFIG['model_name'], num_classes=NUM_CLASSES, pretrained=True).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

    best_val_loss = float('inf')
    best_macro = 0.0
    best_samples = 0.0

    for epoch in range(1, CONFIG['epochs'] + 1):
        train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)
        val_loss, val_macro_f1, val_samples_f1 = validate(model, val_loader, criterion, device)

        print(f"Epoch [{epoch}/{CONFIG['epochs']}] | Train Loss: {train_loss:.4f} | " \
              f"Val Loss: {val_loss:.4f} | Macro F1: {val_macro_f1:.4f} | Samples F1: {val_samples_f1:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_macro = val_macro_f1
            best_samples = val_samples_f1
            torch.save(model.state_dict(), f"best_model_fold_{fold}.pth")

    print(f"\n Fold {fold + 1} 완료 | Best Val Loss: {best_val_loss:.4f}"\
          f"| Best Macro: {best_macro:.4f} | Best Samples: {best_samples:.4f}")
    oof_macro_f1.append(best_macro)
    oof_samples_f1.append(best_samples)

print("\n" + "="*10 + "최종 Out Of Fold 성능 통계" + "="*10)
print(f"Mean Macro F1 Score: {np.mean(oof_macro_f1):.4f}")

```
========================================
 FOLD 1 / 5 가동
========================================
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
WARNING:huggingface_hub.utils._http:Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
model.safetensors: 100%
 46.8M/46.8M [00:01<00:00, 50.5MB/s]
Epoch [1/3] | Train Loss: 0.1270 | Val Loss: 0.0623 | Macro F1: 0.1658 | Samples F1: 0.7324
Epoch [2/3] | Train Loss: 0.0563 | Val Loss: 0.0505 | Macro F1: 0.2397 | Samples F1: 0.7863
Epoch [3/3] | Train Loss: 0.0492 | Val Loss: 0.0472 | Macro F1: 0.2703 | Samples F1: 0.7997

 Fold 1 완료 | Best Val Loss: 0.0472| Best Macro: 0.2703 | Best Samples: 0.7997

========================================
 FOLD 2 / 5 가동
========================================
Epoch [1/3] | Train Loss: 0.1256 | Val Loss: 0.0621 | Macro F1: 0.1777 | Samples F1: 0.7351
Epoch [2/3] | Train Loss: 0.0560 | Val Loss: 0.0508 | Macro F1: 0.2387 | Samples F1: 0.7825
Epoch [3/3] | Train Loss: 0.0488 | Val Loss: 0.0465 | Macro F1: 0.2668 | Samples F1: 0.8036

 Fold 2 완료 | Best Val Loss: 0.0465| Best Macro: 0.2668 | Best Samples: 0.8036

========================================
 FOLD 3 / 5 가동
========================================
Epoch [1/3] | Train Loss: 0.1246 | Val Loss: 0.0628 | Macro F1: 0.1782 | Samples F1: 0.7293
Epoch [2/3] | Train Loss: 0.0555 | Val Loss: 0.0500 | Macro F1: 0.2478 | Samples F1: 0.7889
Epoch [3/3] | Train Loss: 0.0485 | Val Loss: 0.0467 | Macro F1: 0.2641 | Samples F1: 0.8018

 Fold 3 완료 | Best Val Loss: 0.0467| Best Macro: 0.2641 | Best Samples: 0.8018

========================================
 FOLD 4 / 5 가동
========================================
Epoch [1/3] | Train Loss: 0.1245 | Val Loss: 0.0632 | Macro F1: 0.1679 | Samples F1: 0.7237
Epoch [2/3] | Train Loss: 0.0565 | Val Loss: 0.0502 | Macro F1: 0.2411 | Samples F1: 0.7859
Epoch [3/3] | Train Loss: 0.0490 | Val Loss: 0.0469 | Macro F1: 0.2671 | Samples F1: 0.8007

 Fold 4 완료 | Best Val Loss: 0.0469| Best Macro: 0.2671 | Best Samples: 0.8007

========================================
 FOLD 5 / 5 가동
========================================
Epoch [1/3] | Train Loss: 0.1280 | Val Loss: 0.0637 | Macro F1: 0.1722 | Samples F1: 0.7275
Epoch [2/3] | Train Loss: 0.0566 | Val Loss: 0.0512 | Macro F1: 0.2302 | Samples F1: 0.7825
Epoch [3/3] | Train Loss: 0.0494 | Val Loss: 0.0473 | Macro F1: 0.2669 | Samples F1: 0.8012

 Fold 5 완료 | Best Val Loss: 0.0473| Best Macro: 0.2669 | Best Samples: 0.8012

==========최종 Out Of Fold 성능 통계==========
Mean Macro F1 Score: 0.2670
```